# 32. TVM / MLIR Deep Practice | TVM / MLIR 深度实践

**难度：** Hard | **环境：** CPU-first | **标签：** `编译系统`, `TVM/MLIR`, `Lowering` | **目标人群：** 编译与系统入门者

![多层 IR、Pass 与代码生成](../docs/public/01_Hardware_Math_and_Systems/32_ir_pass_codegen.svg)

---

## 本节导读

从高层图到可执行 kernel，中间并不是简单翻译，而是一整条 IR 变换、算子组织、调度和 codegen 的链路。很多编译系统问题之所以难，不在于“写不写得出代码”，而在于同一个计算图怎样被改写成更贴合目标硬件的执行形式。

本节沿着中间表示、变换阶段、硬件约束与后端生成建立编译链路直觉，为后续算子优化中的调度、内核实现和后端选择提供基础。学习重点是判断每一阶段改变了什么，以及目标硬件为何会限制候选优化。

**关键词：** `TVM`, `MLIR`, `Relay`

---

## 前置阅读

**导语：** 这一页先把图优化、算子融合和 lowering 的直觉接上，再看 TVM / MLIR 为什么会在后端链路里占核心位置。

- [09. AI Compilers and Graph Optimization | AI 编译器与计算图优化](./09_AI_Compilers_and_Graph_Optimization.ipynb)
- [19. Operator Fusion Introduction | 算子融合导论](./19_Operator_Fusion_Introduction.ipynb)
- [33. TCO and Cost Model | 算力评估与 TCO 模型](./33_TCO_and_Cost_Model.ipynb)


## Q1：TVM / MLIR 这类编译器后端在做什么？


后端要做的不是“把高层代码换个语法”，而是把高层表示逐步 lowering 到能在目标硬件上高效运行的形式。

这个过程通常会涉及：
- 图或 IR 的转换；
- 算子拆分与融合；
- 调度与布局优化；
- 面向目标 backend 的 codegen。

所以编译器后端本质上是在把“可表达”变成“可高效执行”。


### Q1小验证：lowering 不是翻译

先把“变成能跑”与“变得更快跑”分开理解。

In [ ]:
source_ir = {'op': 'matmul', 'dtype': 'fp16', 'shape': ('B', 4096, 4096)}
shape_env = {'B': 8}

def specialize_shape(graph):
    graph = dict(graph)
    graph['shape'] = tuple(shape_env.get(dim, dim) for dim in graph['shape'])
    graph['shape_specialized'] = True
    return graph

def legalize_ops(graph):
    graph = dict(graph)
    graph['op'] = 'cuda.matmul'
    graph['legalized'] = True
    return graph

def select_schedule(graph):
    graph = dict(graph)
    m, n, k = graph['shape']
    tile = 128
    graph['schedule'] = {'tile': tile, 'warps': 4}
    graph['schedule_valid'] = n % tile == 0 and k % tile == 0
    return graph

def codegen_cuda(graph):
    graph = dict(graph)
    if not graph.get('legalized') or not graph.get('schedule_valid'):
        raise ValueError('IR 尚未满足 CUDA 代码生成约束')
    graph['backend'] = 'cuda'
    return graph

pipeline = [specialize_shape, legalize_ops, select_schedule, codegen_cuda]
ir = source_ir
for step in pipeline:
    ir = step(ir)
    print(step.__name__, '->', ir)

assert source_ir == {'op': 'matmul', 'dtype': 'fp16', 'shape': ('B', 4096, 4096)}
assert ir['backend'] == 'cuda' and ir['shape_specialized'] and ir['schedule_valid']

## Q2：为什么 autotune 和 backend 选择这么重要？


同一个高层表示，在不同硬件、不同 tile、不同 schedule 下，性能可能差别很大。

autotune 的作用，是在可行搜索空间里找更适合目标硬件的参数组合；backend 选择则决定最终会落到哪条执行路径上。

所以编译器后端不是“一次写完就完”，而是需要根据硬件不断调优和选择。


### Q2小验证：为什么参数搜索有价值

不同 tile 和 schedule 会直接改变性能。

In [ ]:
candidates = [
    {'tile': 64, 'warps': 4, 'shared_memory_bytes': 32_768, 'latency_us': 82.4},
    {'tile': 128, 'warps': 4, 'shared_memory_bytes': 65_536, 'latency_us': 54.7},
    {'tile': 256, 'warps': 8, 'shared_memory_bytes': 131_072, 'latency_us': 49.1},
]
problem_size = 4096
shared_memory_limit = 99_000

feasible = [
    row for row in candidates
    if problem_size % row['tile'] == 0
    and row['shared_memory_bytes'] <= shared_memory_limit
]
best = min(feasible, key=lambda row: row['latency_us'])
print('feasible:', feasible)
print('selected:', best)

assert best['tile'] == 128
assert all(row['shared_memory_bytes'] <= shared_memory_limit for row in feasible)

## Q3：为什么编译器和 kernel 实现最后会连到一起？


编译器决定“怎么拆、怎么排、怎么调”，kernel 决定“怎么真的跑”。

一旦进入目标硬件执行层，编译器的调度结果就会具体反映到 kernel 的内存访问、并行组织和算子组合上。反过来，kernel 能力也会限制编译器能把图优化到什么程度。

所以 TVM / MLIR 不是和 Triton、CUDA 分开的知识，而是和它们组成一条从高层到低层的连续链路。


### Q3小验证：编译器和 kernel 谁更靠近硬件

答案不是二选一，而是两者互相约束。

In [ ]:
compiler_passes = [
    ('fusion', {'shared_memory'}),
    ('tensor_core_vectorize', {'mma_fp16'}),
    ('layout_opt', {'contiguous_load'}),
]

hardware_capabilities = {'shared_memory', 'contiguous_load'}
decisions = []

for name, required in compiler_passes:
    missing = sorted(required - hardware_capabilities)
    decisions.append({'pass': name, 'keep': not missing, 'missing': missing})

for row in decisions:
    print(row)

assert next(row for row in decisions if row['pass'] == 'fusion')['keep']
assert next(row for row in decisions if row['pass'] == 'tensor_core_vectorize')['missing'] == ['mma_fp16']

## Q4：为什么 MLIR 更强调多层 IR 和 pass pipeline，而不是一步到位的 lower？


更深的后端优化，往往不是靠一个“万能 lowering”完成的，而是靠一串可组合、可局部验证的 passes 逐层收敛。

这样做有三个好处：
- **可控**：每一层 IR 只处理自己能表达清楚的问题，避免把所有硬件细节一次塞进去；
- **可复用**：同一套上层表示可以映射到不同硬件后端；
- **可优化**：不同 pass 可以分别针对布局、合法化、调度和生成做局部优化。

所以 MLIR 的价值，不只是“中间表示更多”，而是让 lowering 过程本身变成一条可维护的优化链路。


### Q4小验证：为什么分层比一步到位更稳

In [ ]:
pass_dependencies = {
    'shape_specialize': set(),
    'legalize_ops': {'shape_specialize'},
    'layout_transform': {'legalize_ops'},
    'scheduling': {'layout_transform'},
    'codegen': {'scheduling'},
}

def validate_pipeline(pipeline):
    completed = set()
    for name in pipeline:
        missing = pass_dependencies[name] - completed
        if missing:
            return False, {'pass': name, 'missing': sorted(missing)}
        completed.add(name)
    return True, {'completed': pipeline}

valid_pipeline = ['shape_specialize', 'legalize_ops', 'layout_transform', 'scheduling', 'codegen']
invalid_pipeline = ['shape_specialize', 'scheduling', 'legalize_ops', 'layout_transform', 'codegen']
print('valid:', validate_pipeline(valid_pipeline))
print('invalid:', validate_pipeline(invalid_pipeline))

assert validate_pipeline(valid_pipeline)[0]
assert not validate_pipeline(invalid_pipeline)[0]


## ⚠️ 常见误区

- 后端不是简单代码生成。
- autotune 不是可有可无，而是决定性能的重要部分。
- 编译器和 kernel 不是两条独立链路。
- 图到 kernel 的过程本质上是在做层层约束下的优化。

## 相关阅读

**导语：** 如果想继续把编译后端、调度和成本模型串起来，可以接着看这些页。

- [29. CUDA Stream Advanced Scheduling | CUDA Stream 高级调度](./29_CUDA_Stream_Advanced_Scheduling.ipynb)
- [30. Dynamic Shape Handling | 动态 Shape 处理](./30_Dynamic_Shape_Handling.ipynb)
- [04. Triton 矩阵乘法 (GEMM) 与自动调优 (Autotune)](../03_Triton_Kernels/04_Triton_GEMM_Tutorial.ipynb)
---